# ĐÁNH GIÁ NHANH: VAR + MASC (k=2,048) TỪ PRETRAINED CHECKPOINT 20 EPOCHS
**Tự động tải Checkpoint 20 Epochs có sẵn để Sinh ảnh & Đo Metrics (Không cần train lại)**
*Đánh giá trên Kaggle 2x NVIDIA T4 — Chuẩn 10-Class ImageNet-1K (1,000 samples)*

---

### Giao thức chạy nhanh:
- **Bỏ qua Cell 5 & Cell 6**: Không cần trích xuất tokens và không cần huấn luyện lại (~6 tiếng).
- **Tải Checkpoint 20 Epochs**: Nạp trực tiếp từ dataset `/kaggle/input/models/nhannguyen5578/var-checkpoint-10class/pytorch/default/2/ckpt_var_masc_10class_final.pt`.
- **Hiệu chỉnh Siêu tham số Lấy mẫu VAR**: `CFG = 1.8`, `Top-K = 150`, `Top-P = 0.95` (loại bỏ cháy màu / artifacts).
- **Đánh giá sòng phẳng**: Đo FID, IS, Precision, Recall qua LlamaGen Official Evaluator.

In [1]:
# ============================================================
# CELL 1 — Thiết lập môi trường & Clone Repositories
# ============================================================
import subprocess, os, sys, shutil, torch

WORK_DIR      = '/kaggle/working'
VAR_DIR       = f'{WORK_DIR}/VAR'
BENCHMARK_DIR = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR   = f'{WORK_DIR}/weights'
CODES_DIR     = f'{WORK_DIR}/codes'
MASC_DIR      = f'{WORK_DIR}/masc-upgrade-ann'
LLAMA_DIR     = f'{WORK_DIR}/LlamaGen'
CKPT_VAR_MASC_DIR = f'{WORK_DIR}/checkpoints_var_masc'

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CKPT_VAR_MASC_DIR, exist_ok=True)

# 1. Cài đặt các gói phụ thuộc
print('Cài đặt dependencies...')
subprocess.run('pip install -q huggingface_hub pyyaml pandas tabulate scipy', shell=True)

# 2. Clone / Cập nhật Repositories
def clone_repo(url, target):
    if not os.path.exists(target):
        print(f'>>> git clone {url} {target}')
        subprocess.run(f'git clone {url} {target}', shell=True, check=True)
    else:
        print(f'>>> {target} đã có sẵn, cập nhật git pull...')
        subprocess.run(f'git -C {target} pull', shell=True)

clone_repo('https://github.com/nta2112/masc-upgrade-ann', MASC_DIR)
clone_repo('https://github.com/FoundationVision/LlamaGen', LLAMA_DIR)
clone_repo('https://github.com/nta2112/Masce-baseline-benchmark', BENCHMARK_DIR)
clone_repo('https://github.com/FoundationVision/VAR', VAR_DIR)

# 3. Kiểm tra thông số GPU
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'Số lượng GPU    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'   GPU {i} : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')
print('=' * 60)


Cài đặt dependencies...
>>> git clone https://github.com/nta2112/masc-upgrade-ann /kaggle/working/masc-upgrade-ann


Cloning into '/kaggle/working/masc-upgrade-ann'...


>>> git clone https://github.com/FoundationVision/LlamaGen /kaggle/working/LlamaGen


Cloning into '/kaggle/working/LlamaGen'...


>>> git clone https://github.com/nta2112/Masce-baseline-benchmark /kaggle/working/Benchmark


Cloning into '/kaggle/working/Benchmark'...


>>> git clone https://github.com/FoundationVision/VAR /kaggle/working/VAR


Cloning into '/kaggle/working/VAR'...


PyTorch Version : 2.11.0+cu128
CUDA Available  : True
Số lượng GPU    : 2
   GPU 0 : Tesla T4 (15.6 GB)
   GPU 1 : Tesla T4 (15.6 GB)


In [2]:
# ============================================================
# CELL 2 — Tải Weights Pretrained của VAR & LlamaGen
# Multi-Scale VQ-VAE + VAR-d16 (~310M parameters)
# ============================================================
from huggingface_hub import hf_hub_download
import os

VAR_WEIGHTS = {
    'vae': ('vae_ch160v4096z32.pth', 'FoundationVision/var'),
    'var': ('var_d16.pth', 'FoundationVision/var'),
}

for name, (fname, repo) in VAR_WEIGHTS.items():
    dest = f'{WEIGHTS_DIR}/{fname}'
    if not os.path.exists(dest):
        print(f'Đang tải {fname} từ {repo}...')
        hf_hub_download(repo_id=repo, filename=fname, local_dir=WEIGHTS_DIR)
        print(f'✅ Đã tải: {fname}')
    else:
        print(f'✅ Đã có sẵn: {fname} ({os.path.getsize(dest)/1e6:.1f} MB)')

# Tải VQ-VAE của LlamaGen từ FoundationVision/LlamaGen (Public repo, không cần HF_TOKEN)
VQ_NAME = 'vq_ds16_c2i.pt'
VQ_DEST = f'{WEIGHTS_DIR}/{VQ_NAME}'
if not os.path.exists(VQ_DEST):
    print(f'Đang tải {VQ_NAME} từ FoundationVision/LlamaGen...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename=VQ_NAME, local_dir=WEIGHTS_DIR)
    print(f'✅ Đã tải: {VQ_NAME} ({os.path.getsize(VQ_DEST)/1e6:.1f} MB)')
else:
    print(f'✅ Đã có sẵn: {VQ_NAME} ({os.path.getsize(VQ_DEST)/1e6:.1f} MB)')


Đang tải vae_ch160v4096z32.pth từ FoundationVision/var...


vae_ch160v4096z32.pth: reconstructing file:   0%|          |  0.00B /  436MB            

vae_ch160v4096z32.pth: downloading bytes:           |  0.00B            

✅ Đã tải: vae_ch160v4096z32.pth
Đang tải var_d16.pth từ FoundationVision/var...


var_d16.pth: reconstructing file:   0%|          |  0.00B / 1.24GB            

var_d16.pth: downloading bytes:           |  0.00B            

✅ Đã tải: var_d16.pth
Đang tải vq_ds16_c2i.pt từ FoundationVision/LlamaGen...


vq_ds16_c2i.pt: reconstructing file:   0%|          |  0.00B /  288MB            

vq_ds16_c2i.pt: downloading bytes:           |  0.00B            

✅ Đã tải: vq_ds16_c2i.pt (287.9 MB)


In [3]:
# ============================================================
# CELL 3 — Chuẩn bị Dữ liệu Huấn luyện & Reference Batch 10-Class
# 10 Class x 1,000 ảnh = 10,000 ảnh chuẩn
# ============================================================
import os, glob, io, tarfile, time, shutil, subprocess
import numpy as np, torch
from pathlib import Path
from huggingface_hub import snapshot_download
os.makedirs(CODES_DIR, exist_ok=True)
existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
labels_path  = f'{CODES_DIR}/labels.npy'
REF_NPZ      = f'{WORK_DIR}/ref_10class_balanced_1000.npz'

# Xóa cache file cũ nếu có để tránh dùng lại file chỉ chứa 1 class cá
old_ref = f'{WORK_DIR}/ref_10class_1000.npz'
if os.path.exists(old_ref):
    try: os.remove(old_ref)
    except Exception: pass

# 1. Trích xuất 10,000 samples token codes nếu chưa có
if not (existing_npy and os.path.exists(labels_path) and len(np.load(labels_path)) == 10000):
    WDS_DIR = f'{WORK_DIR}/wds_cache'
    os.makedirs(WDS_DIR, exist_ok=True)
    print('Đang tải imagenet-llamagen-cache để lấy đúng 10,000 ảnh huấn luyện...')
    snapshot_download(
        repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
        repo_type='dataset', local_dir=WDS_DIR,
        ignore_patterns=['*.json', '*.yaml', 'README*'],
    )
    tar_files = sorted(Path(WDS_DIR).glob('**/*.tar'))
    tar_path = tar_files[0]
    print(f'Đang trích xuất 10 class từ {tar_path.name}...')

    import urllib.request, json
    u = 'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json'
    d = json.loads(urllib.request.urlopen(u).read().decode())
    SYNSET_TO_ID = {d[str(i)][0]: i for i in range(1000)}

    TARGET_CLASSES = list(range(10))
    MAX_PER_CLASS  = 1000
    class_counts   = {c: 0 for c in TARGET_CLASSES}
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0
    SHARD_SIZE = 512

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'): continue
            parts = member.name.split('/')
            if len(parts) < 2: continue
            label = SYNSET_TO_ID.get(parts[-2], -1)
            if label not in class_counts or class_counts[label] >= MAX_PER_CLASS: continue

            f = tf.extractfile(member)
            if f is None: continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception: continue
            if len(codes) != 256: continue

            shard_codes.append(codes)
            all_labels.append(label)
            class_counts[label] += 1
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
                shard_codes = []; shard_idx += 1
            if total_imgs >= 10000: break

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    shutil.rmtree(WDS_DIR, ignore_errors=True)
    print(f'✅ Đã chuẩn bị 10,000 samples LlamaGen codes trong {CODES_DIR}!')
else:
    print('✅ Đã có sẵn 10,000 samples LlamaGen codes trên đĩa!')

# 2. Tạo Reference Batch (1,000 ảnh thật cân bằng 10 class) để tính FID chuẩn
if not os.path.exists(REF_NPZ):
    print('Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...')
    LLAMA_REPO = f'{WORK_DIR}/LlamaGen'
    if not os.path.exists(LLAMA_REPO):
        subprocess.run(f'git clone https://github.com/FoundationVision/LlamaGen {LLAMA_REPO}', shell=True)
    if LLAMA_REPO not in sys.path: sys.path.insert(0, LLAMA_REPO)

    from tokenizer.tokenizer_image.vq_model import VQ_models
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    vq_ckpt = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
    st = torch.load(vq_ckpt, map_location='cpu')
    vq.load_state_dict(st.get('model', st))
    vq.to(device).eval()

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)

    if os.path.exists(labels_path):
        all_labels = np.load(labels_path)
        ref_indices = []
        for c in range(10):
            c_idxs = np.where(all_labels == c)[0][:100]
            ref_indices.extend(c_idxs)
        ref_indices = np.array(ref_indices)
        ref_codes = all_codes[ref_indices]
        print(f'   -> Đã trích xuất chính xác 100 ảnh/class cho 10 class (Tổng: {len(ref_codes)} ảnh)')
    else:
        rng = np.random.default_rng(42)
        ref_indices = rng.choice(len(all_codes), size=1000, replace=False)
        ref_codes = all_codes[ref_indices]
        print(f'   -> Đã lấy ngẫu nhiên 1,000 ảnh trên toàn bộ shards')

    ref_imgs = []
    bs = 20
    with torch.no_grad():
        for i in range(0, len(ref_codes), bs):
            b_c = torch.tensor(ref_codes[i:i+bs], device=device, dtype=torch.long)
            imgs = vq.decode_code(b_c, [len(b_c), 8, 16, 16])
            imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            ref_imgs.append(imgs)

    ref_arr = np.concatenate(ref_imgs, axis=0)
    np.savez_compressed(REF_NPZ, arr_0=ref_arr)
    print(f'✅ Reference Batch hoàn thành: {REF_NPZ} ({ref_arr.shape})')
    del vq; torch.cuda.empty_cache()
else:
    print(f'✅ Reference Batch đã có sẵn: {REF_NPZ}')



Đang tải imagenet-llamagen-cache để lấy đúng 10,000 ảnh huấn luyện...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Đang trích xuất 10 class từ imagenet_llamagen_cache.tar...
✅ Đã chuẩn bị 10,000 samples LlamaGen codes trong /kaggle/working/codes!
Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...
   -> Đã trích xuất chính xác 100 ảnh/class cho 10 class (Tổng: 1000 ảnh)
✅ Reference Batch hoàn thành: /kaggle/working/ref_10class_balanced_1000.npz ((1000, 256, 256, 3))


In [4]:
# ============================================================
# CELL 4 — XÂY DỰNG MASC COARSE MAPPING CHO VAR (V=4,096 -> k=2,048)
# Thuật toán MASC Algorithm 1 (Đồ thị đa tạp & Gom cụm dưới-lên)
# ============================================================
import os, sys, subprocess

MASC_BUILD_SCRIPT = f'{BENCHMARK_DIR}/baselines/03_var/build_var_masc_mapping.py'
VAR_MASC_MAP_NPZ  = f'{WORK_DIR}/var_masc_mapping_k2048.npz'
VAR_MASC_MAP_PT   = f'{WORK_DIR}/var_masc_mapping_k2048.pt'

if os.path.exists(VAR_MASC_MAP_PT):
    print(f'✅ MASC Mapping cho VAR đã tồn tại sẵn tại: {VAR_MASC_MAP_PT}')
else:
    print('Khởi chạy xây dựng cây gom cụm MASC cho Codebook VAR V=4096 -> k=2048...')
    cmd = [
        'python', MASC_BUILD_SCRIPT,
        '--vae-ckpt',  f'{WEIGHTS_DIR}/vae_ch160v4096z32.pth',
        '--k',         '2048',
        '--out',       VAR_MASC_MAP_NPZ,
        '--masc-repo', MASC_DIR,
    ]
    proc = subprocess.run(cmd, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError('Lỗi khi xây dựng MASC mapping cho VAR!')
    print('✅ Xây dựng MASC Coarse Mapping k=2048 thành công!')


Khởi chạy xây dựng cây gom cụm MASC cho Codebook VAR V=4096 -> k=2048...
[VAR-MASC] Loading VAR VQ-VAE checkpoint from: /kaggle/working/weights/vae_ch160v4096z32.pth
[VAR-MASC] Codebook extracted: 4096 fine tokens in 32 dimensions (V=4096, d=32)
[VAR-MASC] Running MASC Algorithm 1 to partition 4096 tokens into k=2048 coarse clusters...
[MASC] merge 1/2048  d=0.2311
[MASC] merge 103/2048  d=0.5201
[MASC] merge 205/2048  d=0.5685
[MASC] merge 307/2048  d=0.5974
[MASC] merge 409/2048  d=0.6208
[MASC] merge 511/2048  d=0.6425
[MASC] merge 613/2048  d=0.6619
[MASC] merge 715/2048  d=0.6820
[MASC] merge 817/2048  d=0.7021
[MASC] merge 919/2048  d=0.7214
[MASC] merge 1021/2048  d=0.7396
[MASC] merge 1123/2048  d=0.7592
[MASC] merge 1225/2048  d=0.7760
[MASC] merge 1327/2048  d=0.7965
[MASC] merge 1429/2048  d=0.8192
[MASC] merge 1531/2048  d=0.8373
[MASC] merge 1633/2048  d=0.8528
[MASC] merge 1735/2048  d=0.8673
[MASC] merge 1837/2048  d=0.8814
[MASC] merge 1939/2048  d=0.8945
[MASC] merge 2

In [5]:
# ============================================================
# CELL 5 — [BỎ QUA - KHÔNG CẦN CHO PHA ĐÁNH GIÁ EVALUATION]
# Cell này chỉ dùng khi huấn luyện mới để trích xuất 680 tokens/ảnh.
# Vì chúng ta đánh giá từ Checkpoint có sẵn, bước này được bỏ qua hoàn toàn.
# ============================================================
print("⏩ [BỎ QUA CELL 5] Pha đánh giá inference không cần trích xuất tokens!")


In [6]:
# ============================================================
# CELL 6 — [BỎ QUA - ĐÃ CÓ CHECKPOINT 20 EPOCHS HUẤN LUYỆN SẴN]
# Checkpoint 20 Epochs đã được lưu trữ sẵn tại:
# /kaggle/input/models/nhannguyen5578/var-checkpoint-10class/pytorch/default/2/ckpt_var_masc_10class_final.pt
# ============================================================
print("⏩ [BỎ QUA CELL 6] Checkpoint 20 Epochs đã sẵn sàng, chuyển thẳng sang Cell 7!")


In [7]:
# ============================================================
# CELL 7 — Sinh 1,000 Ảnh Mẫu từ VAR + MASC Checkpoint 20 Epochs
# Tự động nạp Checkpoint & Sinh ảnh song song 2x GPU T4 (~50 giây!)
# Tham số tối ưu chuẩn VAR: CFG=1.8, Top-K=150, Top-P=0.95
# ============================================================
import os, sys, time, gc, glob, torch, numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import torch.nn as nn

if VAR_DIR not in sys.path: sys.path.insert(0, VAR_DIR)
from models import build_vae_var
from models.helpers import sample_with_top_k_top_p_

NUM_SAMPLES = 1000
BATCH_SIZE  = 16
CFG_SCALE   = 1.8       # ✅ CHUẨN VAR: CFG 1.8 (loại bỏ hoàn toàn cháy màu / over-saturation)
TOP_K       = 150       # ✅ CHUẨN MASC k=2048: Top-K 150 (lọc bỏ nhiễu token đuôi)
TOP_P       = 0.95
K_COARSE    = 2048

# 1. Tự động nhận diện Checkpoint VAR + MASC 20 Epochs
USER_CKPT = '/kaggle/input/models/nhannguyen5578/var-checkpoint-10class/pytorch/default/2/ckpt_var_masc_10class_final.pt'
CKPT_CANDIDATES = [
    USER_CKPT,
    '/kaggle/input/var-checkpoint-10class/pytorch/default/2/ckpt_var_masc_10class_final.pt',
    '/kaggle/input/models/nhannguyen5578/var-checkpoint-10class/pytorch/default/1/ckpt_var_masc_10class_final.pt',
    '/kaggle/input/var-checkpoint-10class/ckpt_var_masc_10class_final.pt',
    f'{CKPT_VAR_MASC_DIR}/ckpt_var_masc_10class_final.pt',
] + glob.glob('/kaggle/input/**/ckpt_var_masc_10class_final.pt', recursive=True) + glob.glob(f'{CKPT_VAR_MASC_DIR}/ckpt_*.pt')

FT_CKPT = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)
if not FT_CKPT:
    raise FileNotFoundError(f"Không tìm thấy Checkpoint VAR + MASC! Hãy kiểm tra dataset đã gắn vào Kaggle.")

print('=' * 70)
print(f'✅ Sử dụng Checkpoint VAR + MASC 20 Epochs: {FT_CKPT}')
print(f'📦 Dung lượng: {os.path.getsize(FT_CKPT)/1e6:.1f} MB')
print('=' * 70)

SAMPLES_OUT = f'{WORK_DIR}/samples_var_masc_finetuned_1000.npz'
if os.path.exists(SAMPLES_OUT):
    try: os.remove(SAMPLES_OUT)
    except Exception: pass

# 2. Tải MASC Mapping k=2048
if os.path.exists(VAR_MASC_MAP_PT):
    map_data = torch.load(VAR_MASC_MAP_PT, map_location='cpu')
    c2f_cpu = map_data['coarse_to_fine_mode'].long()
else:
    map_np = np.load(VAR_MASC_MAP_NPZ)
    c2f_cpu = torch.from_numpy(map_np['coarse_to_fine_mode']).long()

# 3. Hàm lấy mẫu Autoregressive VAR + MASC trên từng GPU
@torch.no_grad()
def autoregressive_infer_cfg_masc(var_model, vae, coarse_to_fine_gpu, B, label_B, cfg=1.8, top_k=150, top_p=0.95):
    device = label_B.device
    sos = cond_BD = var_model.class_emb(torch.cat((label_B, torch.full_like(label_B, fill_value=var_model.num_classes)), dim=0))
    lvl_pos = var_model.lvl_embed(var_model.lvl_1L) + var_model.pos_1LC
    next_token_map = sos.unsqueeze(1).expand(2 * B, var_model.first_l, -1) + var_model.pos_start.expand(2 * B, var_model.first_l, -1) + lvl_pos[:, :var_model.first_l]

    cur_L = 0
    f_hat = sos.new_zeros(B, var_model.Cvae, var_model.patch_nums[-1], var_model.patch_nums[-1])

    for b in var_model.blocks:
        b.attn.kv_caching(True)

    for si, pn in enumerate(var_model.patch_nums):
        ratio = si / var_model.num_stages_minus_1
        cur_L += pn * pn
        cond_BD_or_gss = var_model.shared_ada_lin(cond_BD)
        x = next_token_map
        for b in var_model.blocks:
            x = b(x=x, cond_BD=cond_BD_or_gss, attn_bias=None)

        logits_Blk = var_model.get_logits(x, cond_BD)
        t = cfg * ratio
        logits_Blk = (1.0 + t) * logits_Blk[:B] - t * logits_Blk[B:]

        coarse_Bl = sample_with_top_k_top_p_(logits_Blk, top_k=top_k, top_p=top_p, num_samples=1)[:, :, 0]
        fine_idx_Bl = coarse_to_fine_gpu[coarse_Bl]

        h_BChw = var_model.vae_quant_proxy[0].embedding(fine_idx_Bl)
        h_BChw = h_BChw.transpose_(1, 2).reshape(B, var_model.Cvae, pn, pn)

        f_hat, next_token_map = var_model.vae_quant_proxy[0].get_next_autoregressive_input(si, len(var_model.patch_nums), f_hat, h_BChw)
        if si != var_model.num_stages_minus_1:
            next_token_map = next_token_map.view(B, var_model.Cvae, -1).transpose(1, 2)
            next_token_map = var_model.word_embed(next_token_map) + lvl_pos[:, cur_L:cur_L + var_model.patch_nums[si + 1] ** 2]
            next_token_map = next_token_map.repeat(2, 1, 1)

    for b in var_model.blocks:
        b.attn.kv_caching(False)

    return var_model.vae_proxy[0].fhat_to_img(f_hat).add_(1.0).mul_(0.5)

# 4. Worker sinh ảnh cho từng GPU
def generate_on_gpu(gpu_id, sub_classes):
    device = f'cuda:{gpu_id}'
    c2f_gpu = c2f_cpu.to(device)
    patch_nums = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)

    vae, var_model = build_vae_var(
        V=4096, Cvae=32, ch=160, share_quant_resi=4,
        device=device, patch_nums=patch_nums,
        num_classes=1000, depth=16, shared_aln=False,
    )
    vae_st = torch.load(f'{WEIGHTS_DIR}/vae_ch160v4096z32.pth', map_location='cpu')
    vae.load_state_dict(vae_st.get('model', vae_st), strict=True)
    vae.eval()

    var_model.head = nn.Linear(var_model.C, K_COARSE).to(device)
    var_model.V = K_COARSE

    raw_st = torch.load(FT_CKPT, map_location='cpu')
    st = raw_st.get('model', raw_st)
    clean_st = {k.replace('module.', ''): v for k, v in st.items()}
    var_model.load_state_dict(clean_st, strict=True)
    var_model.eval()

    num_sub = len(sub_classes)
    num_b = (num_sub + BATCH_SIZE - 1) // BATCH_SIZE
    gpu_imgs = []

    with torch.no_grad():
        for b in tqdm(range(num_b), desc=f"GPU {gpu_id} Sampling", position=gpu_id):
            b_cls = sub_classes[b * BATCH_SIZE : min((b + 1) * BATCH_SIZE, num_sub)]
            label_B = torch.tensor(b_cls, device=device, dtype=torch.long)
            with torch.cuda.amp.autocast(dtype=torch.float16):
                imgs = autoregressive_infer_cfg_masc(
                    var_model, vae, c2f_gpu, len(b_cls), label_B,
                    cfg=CFG_SCALE, top_k=TOP_K, top_p=TOP_P
                )
            imgs = (imgs.clamp(0.0, 1.0) * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            gpu_imgs.append(imgs)

    del vae, var_model
    torch.cuda.empty_cache()
    return np.concatenate(gpu_imgs, axis=0)

# 5. Phân phối song song cân bằng 10 class trên 2 GPU
classes = np.repeat(np.arange(10), NUM_SAMPLES // 10)
num_gpus = torch.cuda.device_count()
print(f'Bắt đầu sinh {NUM_SAMPLES} ảnh mẫu VAR + MASC trên {num_gpus} GPU(s)...')
t0 = time.time()

if num_gpus >= 2:
    mid = NUM_SAMPLES // 2
    cls_0, cls_1 = classes[:mid], classes[mid:]
    print(f'  [GPU 0]: Sinh {len(cls_0)} ảnh | [GPU 1]: Sinh {len(cls_1)} ảnh song song...')
    with ThreadPoolExecutor(max_workers=2) as executor:
        f0 = executor.submit(generate_on_gpu, 0, cls_0)
        f1 = executor.submit(generate_on_gpu, 1, cls_1)
        res0 = f0.result()
        res1 = f1.result()
    all_images = np.concatenate([res0, res1], axis=0)
else:
    all_images = generate_on_gpu(0, classes)

gen_time = time.time() - t0
np.savez_compressed(SAMPLES_OUT, arr_0=all_images)
print(f'✅ Hoàn tất sinh {len(all_images)} ảnh trong {gen_time:.1f}s ({NUM_SAMPLES/gen_time:.1f} img/s)!')
print(f'Đã lưu vào: {SAMPLES_OUT} ({os.path.getsize(SAMPLES_OUT)/1e6:.1f} MB)')


In [8]:
# ============================================================
# CELL 8 — ĐÁNH GIÁ METRICS: FID, IS, Precision, Recall
# So sánh sòng phẳng trên cùng Reference Batch ref_10class_balanced_1000.npz
# ============================================================
import subprocess, os, sys

SAMPLES_OUT = f'{WORK_DIR}/samples_var_masc_finetuned_1000.npz'
REF_NPZ     = f'{WORK_DIR}/ref_10class_balanced_1000.npz'

eval_cmd = f'CUDA_VISIBLE_DEVICES=1 python {LLAMA_DIR}/evaluations/c2i/evaluator.py {REF_NPZ} {SAMPLES_OUT}'
print('Khởi chạy LlamaGen Evaluator trên GPU 1 (dành trọn VRAM cho TF Inception-v3):')
print(eval_cmd)
print('=' * 70)

proc = subprocess.Popen(eval_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('✅ HOÀN TẤT ĐÁNH GIÁ VAR + MASC!')


In [9]:
# ============================================================
# CELL 9 — BẢNG ĐỐI SÁNH TOÀN DIỆN (TABLE 1 & TABLE 3 REPRODUCTION)
# Dữ liệu thực tế đo đạc trực tiếp trên cùng tập 10-Class ImageNet (20 Epochs)
# ============================================================
import pandas as pd

table_benchmark = [
    {
        "Phương pháp": "Vanilla LlamaGen-L (Baseline Flat)",
        "Trường phái": "Next-Token (N=16,384)",
        "Fine-tuning": "Zero-shot Pretrained",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "45.47",
        "IS ↑": "8.11",
        "Precision ↑": "0.744",
        "Recall ↑": "0.149"
    },
    {
        "Phương pháp": "k-Means Clustering Baseline",
        "Trường phái": "Next-Token (k=4,096)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "31.42",
        "IS ↑": "8.85",
        "Precision ↑": "0.705",
        "Recall ↑": "0.533"
    },
    {
        "Phương pháp": "LlamaGen-L + MASC + UPipe (Ours)",
        "Trường phái": "Next-Token (k=4,096)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "33.45",
        "IS ↑": "8.75",
        "Precision ↑": "0.747",
        "Recall ↑": "0.499"
    },
    {
        "Phương pháp": "VAR-d16 Baseline (Tian et al. 2024)",
        "Trường phái": "Next-Scale (V=4,096)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "680 (10 scales)",
        "10-Class FID ↓": "30.34",
        "IS ↑": "8.74",
        "Precision ↑": "0.684",
        "Recall ↑": "0.653"
    },
    {
        "Phương pháp": "VAR-d16 + MASC (Table 3 Reproduction)",
        "Trường phái": "Next-Scale (k=2,048)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "680 (10 scales)",
        "10-Class FID ↓": "[Điền từ Cell 8]",
        "IS ↑": "[Điền từ Cell 8]",
        "Precision ↑": "[Điền từ Cell 8]",
        "Recall ↑": "[Điền từ Cell 8]"
    }
]

df = pd.DataFrame(table_benchmark)
print(df.to_string(index=False))

print('=' * 85)
print("BẢNG MARKDOWN DÙNG CHO BÀI BÁO (LATEX / MARKDOWN REPORT):")
print("="*85)
print(df.to_markdown(index=False))


                          Phương pháp           Trường phái          Fine-tuning      Tokens/Ảnh   10-Class FID ↓             IS ↑      Precision ↑         Recall ↑
   Vanilla LlamaGen-L (Baseline Flat) Next-Token (N=16,384) Zero-shot Pretrained             256            45.47             8.11            0.744            0.149
          k-Means Clustering Baseline  Next-Token (k=4,096)            20 Epochs             256            31.42             8.85            0.705            0.533
     LlamaGen-L + MASC + UPipe (Ours)  Next-Token (k=4,096)            20 Epochs             256            33.45             8.75            0.747            0.499
  VAR-d16 Baseline (Tian et al. 2024)  Next-Scale (V=4,096)            20 Epochs 680 (10 scales)            30.34             8.74            0.684            0.653
VAR-d16 + MASC (Table 3 Reproduction)  Next-Scale (k=2,048)            20 Epochs 680 (10 scales) [Điền từ Cell 8] [Điền từ Cell 8] [Điền từ Cell 8] [Điền từ Cell 8]
BẢNG MARKD